# B-Free x GlobalForge — Notebook 01: Setup Bundle (K1)

Cài đặt môi trường **offline** cho Kaggle (RTX PRO 6000 Blackwell 96GB, sm_120, cu128, Python 3.11), clone repo tích hợp và validate.

**Yêu cầu Kaggle input:** dataset `bfree-wheels` — wheel bundle chứa `torch==2.8.0+cu128`, `torchvision==0.23.0+cu128`, `timm==1.0.22`, `peft==0.15.2`, `transformers==4.55.4`, `pandas==2.3.3`, `numpy==1.26.4`, `matplotlib==3.11.1`, `seaborn==0.13.2`, `scikit-learn`, `scipy`, `Pillow`, `pyyaml`, `tqdm`, `safetensors` (+ dependencies: `huggingface_hub`, `tokenizers`, ...). Tạo bundle trên máy có internet: `pip download -d bfree-wheels/ <các package như trên>` rồi upload làm Kaggle Dataset.

**Pin stack (locked, plan.md):** torch 2.8.0+cu128 (sm_120 cần >=2.8) / pandas 2.3.3 (MUST <3, breaks sklearn nếu >=3) / numpy 1.26.4.

> Hoàn toàn offline: KHÔNG apt-get, KHÔNG internet.pip, KHÔNG HF hub download trong runtime.

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
import os, sys, glob

WHEELS_DIR = "/kaggle/input/bfree-wheels"
assert os.path.isdir(WHEELS_DIR), (
    f"Wheel bundle not found at {WHEELS_DIR}. "
    "Attach the 'bfree-wheels' Kaggle dataset before running this notebook."
)
wheels = sorted(glob.glob(os.path.join(WHEELS_DIR, "*.whl")))
print(f"Found {len(wheels)} wheels in {WHEELS_DIR}")
assert wheels, "No .whl files found in the wheel bundle."

!pip install --no-index --find-links={WHEELS_DIR} \
    torch==2.8.0+cu128 torchvision==0.23.0+cu128
!pip install --no-index --find-links={WHEELS_DIR} \
    timm==1.0.22 peft==0.15.2 transformers==4.55.4 \
    pandas==2.3.3 numpy==1.26.4 matplotlib==3.11.1 seaborn==0.13.2 \
    scikit-learn scipy pyyaml pillow tqdm safetensors

In [ ]:
REPO_URL = "https://github.com/P-Bao/B-Free.git"
REPO_DIR = "/kaggle/working/B-Free"
BRANCH = "integration/loss-backbone"

import os, sys, glob
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already cloned.")

assert os.path.isfile(os.path.join(REPO_DIR, "code", "networks", "bfree_globalforge_vit.py")), "Clone failed: backbone file missing."
stubs = glob.glob(os.path.join(REPO_DIR, "code", "modules", "*_stub.py"))
assert not stubs, f"Stub files still present (K0 not merged?): {stubs}"
print("OK: repo cloned on integration/loss-backbone, no stub files (K0 verified).")

In [ ]:
import sys

sys.path.insert(0, os.path.join(REPO_DIR, "code"))

import torch
import torchvision
import timm
import peft
import transformers
import pandas
import numpy
import sklearn
import yaml
import PIL

print(f"torch        = {torch.__version__}")
print(f"torchvision  = {torchvision.__version__}")
print(f"timm         = {timm.__version__}")
print(f"peft         = {peft.__version__}")
print(f"transformers = {transformers.__version__}")
print(f"pandas       = {pandas.__version__}")
print(f"numpy        = {numpy.__version__}")

assert pandas.__version__.split('.')[0] == '2', "pandas>=3 breaks sklearn — must pin pandas<3 (risk table)."

from networks.bfree_globalforge_vit import BFreeGlobalForgeViT
from configs.loader import load_config, build_model_kwargs
from datasets.bfree_dataset import BFreeDataset, DegradationPipeline
from modules.lib_adapter import LIBAdapter
from modules.gsr_adapter import GSRAdapter
from modules.dcs_loss import DCSLoss, info_nce_loss

cfg = load_config(os.path.join(REPO_DIR, "code", "configs", "bfree_dcs.yaml"))
kwargs = build_model_kwargs(cfg)
print("\nModel kwargs from bfree_dcs.yaml:")
for k, v in kwargs.items():
    print(f"  {k} = {v}")
print("\nIMPORT VALIDATION OK")

In [ ]:
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "No CUDA device — this notebook requires the Kaggle GPU."

props = torch.cuda.get_device_properties(0)
total_mem = getattr(props, "total_mem", None) or getattr(props, "total_memory")
print(f"GPU            : {torch.cuda.get_device_name(0)}")
print(f"VRAM           : {total_mem / 1024**3:.1f} GB")
print(f"Compute caps   : sm_{props.major}{props.minor} (Blackwell = sm_120, needs torch>=2.8 cu128)")
print(f"torch CUDA     : {torch.version.cuda}")
print(f"bf16 supported : {torch.cuda.is_bf16_supported()} (must be True — fp16 forbidden on Blackwell)")

model_smoke = BFreeGlobalForgeViT(img_size=224, pretrained=False)
x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    out = model_smoke(x)
print(f"\nCPU smoke test: logits {tuple(out['logits'].shape)}, cls {tuple(out['cls'].shape)}")
assert out["logits"].shape == (2, 2) and out["cls"].shape == (2, 768)
del model_smoke, x, out
print("GPU + MODEL SMOKE OK")

## Environment Ready (K1 pass)

- Deps installed offline từ wheel bundle (pin stack đúng, pandas<3).
- Repo `P-Bao/B-Free` branch `integration/loss-backbone` cloned, không còn stub files (K0).
- `BFreeGlobalForgeViT` + LIB/GSR/DCS import thành công; forward smoke pass.
- GPU Blackwell sm_120 + bf16 khả dụng, torch 2.8.0+cu128.

**Tiếp theo:** chạy `02_bfree_kaggle_train.ipynb` (train LoRA r=16, 8 epochs @504px).